# DS317 – Lab 3 | NOTEBOOK CHUNG (Nhóm 5 – DS317.R11)
**Đề tài:** Dự báo doanh số và cảnh báo rủi ro tồn kho theo SKU–tuần cho chuỗi bán lẻ điện tử bằng mô hình Transformer đa nhiệm

Notebook này là **notebook tổng**: chỉ *nạp file xuất* từ các notebook khảo sát/tạo nhãn, rồi làm các việc còn lại: **lọc – nối – chia dữ liệu → baseline → Transformer đơn nhiệm/đa nhiệm → độ đo nghiệp vụ → ngưỡng chấp nhận → xuất DataInventory → biểu đồ**.

| Ký hiệu | Ý nghĩa |
|---|---|
| 🟦 **TODO** | Chỗ nhóm phải điền/sửa (đường dẫn, link, ngưỡng, tên cột…) |
| 🎬 **DEMO** | Cell/phần dùng để trình bày ở buổi 6 |
| ⏱ **NẶNG** | Cell chạy lâu → bỏ qua khi demo, chạy trước và lưu kết quả |
| ✅ **KIỂM TRA** | Cell kiểm tra/chống rò rỉ, không được bỏ |

### Bản đồ DEMO (buổi 6)
| Demo | Nội dung | Mục trong notebook |
|---|---|---|
| 🎬 DEMO-1 | **Khảo sát dữ liệu** (bắt buộc theo hướng dẫn): bảng tổng quan, chất lượng, biểu đồ xu hướng/mùa vụ, phân bố nhãn | Mục 4 (hiển thị kết quả 2 notebook) + Mục 6 |
| 🎬 DEMO-2 | **Chia dữ liệu theo thời gian** + ngưỡng nhãn lấy từ train | Mục 5.4–5.6 |
| 🎬 DEMO-3 | **Hỗ trợ quyết định**: Top-K SKU rủi ro cao của một tuần | Mục 9.3 |

> Quy tắc chạy: **Runtime → Run all** phải chạy sạch từ trên xuống. Cell ⏱ NẶNG có cờ `RUN_HEAVY` để bật/tắt.

## 1. Thông tin dự án và liên kết notebook  *(🟦 TODO: điền link)*

| # | Notebook | Nội dung | Link Colab/Drive | File xuất ra (vào `interim/`) | Phụ trách | Trạng thái |
|---|---|---|---|---|---|---|
| NB1 | **ES – bán hàng** (khảo sát + tạo nhãn doanh số) | Khảo sát 27 bảng, chất lượng 6 chiều, SKU–tuần, `y_qty`, đặc trưng doanh số | 🟦 `https://colab.research.google.com/...` | `es_sales_wk`, `es_quality_sales`, `es_table_catalog`, `es_filter_log` | 🟦 | 🟦 |
| NB2 | **ES – tồn kho** (khảo sát + tạo nhãn rủi ro) | `stock_end`, `cov_past`, `cov_fwd`, `trend4` (**chưa gán lớp**) | 🟦 `https://colab.research.google.com/...` | `es_stock_wk`, `es_quality_inventory` | 🟦 | 🟦 |
| NB3 *(tùy chọn)* | Instacart – khảo sát và kiểm tra khả thi | Bằng chứng loại Instacart | 🟦 | `ic_summary`, `ic_feasibility` | 🟦 | 🟦 |
| NB-Tổng | Notebook này | Mục 5 → 13 | 🟦 | `mt_*`, `results/`, `figures/`, `excel/` | 🟦 | 🟦 |

**Dữ liệu và tài liệu:** thư mục Drive dữ liệu gốc 🟦 `...` · README 🟦 `...` · Bản đề xuất/Thuyết minh 🟦 `...`

**Nguồn dữ liệu (A1 – thống nhất ngày truy cập):**

| Nguồn | Giấy phép | Link | Ngày truy cập | Vai trò |
|---|---|---|---|---|
| Electronics Sales Dataset (fahdelatiby) | CC0 | https://www.kaggle.com/datasets/fahdelatiby/electronics-sales-dataset-b-tech-style | 🟦 `dd/mm/yyyy` | Nguồn chính – dùng cho mô hình |
| Instacart Market Basket Analysis (psparks) | CC0 | https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis | 🟦 `dd/mm/yyyy` | Chỉ khảo sát/đối chứng khả thi |

## 2. Quy tắc chung *(áp dụng cho **mọi** notebook của nhóm)*

### 2.1 Quy tắc đặt tên

| Đối tượng | Quy tắc | Ví dụ |
|---|---|---|
| Tiền tố nguồn | `es_` = Electronics Sales · `ic_` = Instacart · `mt_` = modeling table (bảng mô hình) | `es_sales_wk`, `mt_train` |
| Hậu tố giai đoạn | `_raw` vừa đọc · `_clean` sau lọc · `_wk` đã gộp theo tuần | `es_sales_order_lines_raw` |
| Cột khóa | `sku_id`, `week_start` (thứ Hai của tuần, kiểu datetime) | |
| Tên cột | `snake_case`, không dấu, không khoảng trắng | `qty_rmean4`, `cov_fwd` |
| Nhãn | `y_qty` (doanh số tuần t+1) · `cov_fwd` (Coverage nhìn trước) · `risk` (0=Low,1=Medium,2=High) · `risk_name` | |
| Tập chia | `mt_train`, `mt_val`, `mt_test` (biến) · cột `split` ∈ {train,val,test} | |
| Hằng số | `VIẾT_HOA_GẠCH_DƯỚI`, khai báo **một lần** ở ô cấu hình | `HORIZON_WEEKS` |
| Hàm | `động_từ_đối_tượng` snake_case | `make_time_split`, `eval_reg` |
| Kết quả | `res_reg`, `res_clf` (DataFrame), `fig_*` (hình), file hình `fig_<mô_tả>.png` | |
| Mô hình | `S*` = dự báo doanh số · `R*` = rủi ro · `P1` = pipeline hai bước · `MT` = đa nhiệm | `S1_ma4`, `R1_rule` |

### 2.2 Quy tắc dữ liệu (không được vi phạm)
1. **Không rò rỉ:** đặc trưng chỉ dùng thông tin ≤ tuần *t*; nhãn (`y_qty`, `cov_fwd`, `risk`) **không** nằm trong đầu vào.
2. **Fit trên train:** ngưỡng nhãn P60/P85, scaler, encoder, giá trị điền thiếu → chỉ tính trên `mt_train`.
3. **Gán lớp rủi ro làm *sau* khi chia** dữ liệu (NB2 chỉ xuất `cov_fwd` liên tục).
4. **Chia theo thời gian** (mốc tuần chung cho mọi SKU), có **gap 4 tuần**; test chỉ chạy một lần ở cuối.
5. **Một định nghĩa duy nhất** cho: tuần (`week_start`), đơn hợp lệ (`is_valid_order`), cửa sổ thời gian (`SYNC_START/END`).
6. Mọi file trung gian có **khóa `(sku_id, week_start)` không trùng**.

### 2.3 Hợp đồng file giữa notebook

| File (trong `interim/`) | Do | Cột bắt buộc | Cột tùy chọn |
|---|---|---|---|
| `es_sales_wk` | NB1 | `sku_id, week_start, qty, y_qty` | `revenue, avg_price, discount_rate, promo_flag, qty_lag1/2/4/8, qty_rmean4/8, qty_rstd4/8, qty_slope4, category, brand, product_age_weeks` |
| `es_stock_wk` | NB2 | `sku_id, week_start, stock_end, cov_past, cov_fwd, n_fwd_weeks` | `stock_delta, trend4, unit_cost` |

In [ ]:
# ============================ Ô CẤU HÌNH CHUNG (khai báo MỘT LẦN) ============================
import os, sys, json, math, random, warnings, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)

# ---- 🟦 TODO: chỉnh sau khi có kết quả A2 (khảo sát) ----
SEED              = 42
HORIZON_WEEKS     = 4                 # nhãn rủi ro nhìn trước 4 tuần
EPS               = 1e-6              # chặn mẫu số của Coverage
SYNC_START        = '2023-07-03'      # 🟦 đầu cửa sổ chung (thứ Hai)  ~ 07/2023
SYNC_END          = '2025-12-29'      # 🟦 cuối cửa sổ chung            ~ 12/2025
MIN_HISTORY_WEEKS = 26                # 🟦 SKU phải có >= N tuần trong TRAIN
SPLIT_RATIOS      = (0.70, 0.15, 0.15)
GAP_WEEKS         = 4                 # khoảng trống giữa các tập
RISK_PCT_MED      = 60                # ngưỡng P60 (chỉ tính trên train)
RISK_PCT_HIGH     = 85                # ngưỡng P85 (chỉ tính trên train)
KEY               = ['sku_id', 'week_start']
RISK_NAMES        = {0: 'Low', 1: 'Medium', 2: 'High'}

# ---- Cờ điều khiển ----
USE_SYNTHETIC = False   # True = chạy thử KHUNG bằng dữ liệu giả (smoke test), KHÔNG dùng khi nộp
RUN_HEAVY     = False   # True = chạy các cell ⏱ NẶNG (Transformer, lưới siêu tham số)
MOUNT_DRIVE   = True    # mount Google Drive khi chạy trên Colab

# ---- Đường dẫn ----
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/DS317_Nhom5') if IN_COLAB else Path('./DS317_Nhom5')   # 🟦 sửa nếu khác
DIRS = {k: ROOT / k for k in ['raw', 'interim', 'processed', 'results', 'figures', 'excel']}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

# ---- Danh sách cột ----
FEATURE_GROUPS = {
    'sales'      : ['qty','qty_lag1','qty_lag2','qty_lag4','qty_lag8','qty_rmean4','qty_rmean8','qty_rstd4','qty_rstd8','qty_slope4'],
    'price_promo': ['avg_price','discount_rate','promo_flag'],
    'inventory'  : ['stock_end','stock_delta','cov_past'],
    'product'    : ['product_age_weeks'],
    'time'       : ['week_of_year','month'],
}
CAT_COLS  = ['category','brand']                       # 🟦 sửa theo cột thật (không có thì bỏ qua)
LEAK_COLS = ['y_qty','cov_fwd','risk','risk_name','n_fwd_weeks']   # TUYỆT ĐỐI không đưa vào đầu vào
SCHEMA = {
    'es_sales_wk': {'required': ['sku_id','week_start','qty','y_qty'],
                    'optional': sum([FEATURE_GROUPS[g] for g in ['price_promo','product']],[]) + ['revenue','category','brand'] + FEATURE_GROUPS['sales'][1:]},
    'es_stock_wk': {'required': ['sku_id','week_start','stock_end','cov_past','cov_fwd','n_fwd_weeks'],
                    'optional': ['stock_delta','trend4','unit_cost']},
}

# ============================ HÀM DÙNG CHUNG ============================
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch; torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except Exception: pass
set_seed()

def save_df(df, name, folder='interim'):
    p = DIRS[folder] / name
    try:
        df.to_parquet(p.with_suffix('.parquet'), index=False); return p.with_suffix('.parquet')
    except Exception:
        df.to_csv(p.with_suffix('.csv'), index=False); return p.with_suffix('.csv')

def load_df(name, folder='interim'):
    for ext in ('.parquet', '.csv'):
        p = (DIRS[folder] / name).with_suffix(ext)
        if p.exists():
            df = pd.read_parquet(p) if ext == '.parquet' else pd.read_csv(p)
            if 'week_start' in df.columns: df['week_start'] = pd.to_datetime(df['week_start'])
            return df
    return None

def validate_schema(df, name):
    sc = SCHEMA[name]; miss_req = [c for c in sc['required'] if c not in df.columns]
    miss_opt = [c for c in sc['optional'] if c not in df.columns]
    dup = int(df.duplicated(KEY).sum()) if all(k in df.columns for k in KEY) else -1
    print(f'[{name}] shape={df.shape} | thiếu cột BẮT BUỘC: {miss_req or "không"} | khóa trùng: {dup}')
    if miss_opt: print('   cột tùy chọn chưa có:', miss_opt)
    assert not miss_req, f'{name} thiếu cột bắt buộc: {miss_req}'
    assert dup == 0, f'{name} có khóa (sku_id, week_start) trùng'

def describe_df(df, name):
    print(f'== {name}: {df.shape[0]:,} dòng × {df.shape[1]} cột | SKU={df.sku_id.nunique():,} | '
          f'tuần {df.week_start.min().date()} → {df.week_start.max().date()} ({df.week_start.nunique()} tuần)')

STEP_LOG = []
def log_step(df, step):
    STEP_LOG.append({'bước': step, 'số_dòng': len(df), 'số_SKU': df.sku_id.nunique(), 'số_tuần': df.week_start.nunique()})
    print(f'  [{step}] dòng={len(df):,} SKU={df.sku_id.nunique():,} tuần={df.week_start.nunique()}')

def week_start_of(s):                      # ĐỊNH NGHĨA TUẦN DUY NHẤT: thứ Hai
    s = pd.to_datetime(s); return (s - pd.to_timedelta(s.dt.weekday, unit='D')).dt.normalize()

def is_valid_order(df, status_col='OrderStatus', bad=('Cancelled','Canceled','Failed')):   # 🟦 TODO: điền giá trị thật từ A2
    return ~df[status_col].isin(bad)

def save_fig(fig, name):
    p = DIRS['figures'] / f'fig_{name}.png'; fig.savefig(p, dpi=200, bbox_inches='tight'); print('đã lưu', p)

print('Cấu hình xong | ROOT =', ROOT.resolve())

## 3. Môi trường và import dataset gốc

> Notebook tổng **không tạo lại** bảng bán hàng/tồn kho (đã có từ NB1, NB2). Dữ liệu gốc chỉ nạp cho các bảng tra cứu cần thêm (ví dụ `Products`, bảng giá/khuyến mãi) hoặc để kiểm tra chéo.

In [ ]:
# ---- 3.1 Cài thư viện (Colab) ----
# !pip -q install lightgbm pyarrow openpyxl
import importlib
for m in ['sklearn','lightgbm','matplotlib','pyarrow','openpyxl','torch']:
    try: importlib.import_module(m); print(f'{m:12s} OK')
    except Exception: print(f'{m:12s} CHƯA CÀI')

In [ ]:
# có thể thay code bằng import file zip
# ---- 3.2 Import dataset gốc (🟦 TODO: điền tên file đúng của Electronics Sales trong thư mục raw/) ----
# Cách 1: tải thủ công từ Kaggle (link ở Mục 1) rồi để file .csv vào  ROOT/raw/
# Cách 2: dùng Kaggle API (cần kaggle.json):  !kaggle datasets download -d fahdelatiby/electronics-sales-dataset-b-tech-style -p {DIRS['raw']} --unzip
RAW_FILES = {                    # 🟦 tên logic : tên file trong raw/
    'products' : 'Products.csv',
    # 'promotions': 'Promotions.csv',
}
def read_raw(key, usecols=None, **kw):
    p = DIRS['raw'] / RAW_FILES[key]
    if not p.exists():
        print(f'  (chưa có {p.name} – bỏ qua)'); return None
    df = pd.read_csv(p, usecols=usecols, **kw); print(f'  {key}: {df.shape}'); return df

raw = {k: read_raw(k) for k in RAW_FILES}

## 4. Import kết quả của 2 notebook khảo sát/tạo nhãn  🎬 DEMO-1

Nạp các file xuất từ **NB1** (`es_sales_wk`) và **NB2** (`es_stock_wk`) trong `interim/`, kiểm tra hợp đồng file (Mục 2.3), rồi hiển thị tổng quan để demo phần khảo sát dữ liệu.

In [ ]:
# ---- 4.0 (TÙY CHỌN) Chạy thử khung bằng dữ liệu giả — chỉ để kiểm tra notebook chạy được ----
def make_synthetic_inputs(n_sku=60, n_weeks=130, seed=SEED):
    rng = np.random.default_rng(seed)
    weeks = pd.date_range(SYNC_START, periods=n_weeks + HORIZON_WEEKS, freq='W-MON')
    rows = []
    for s in range(n_sku):
        base = rng.uniform(3, 40); trend = rng.normal(0, 0.03)
        t = np.arange(len(weeks)); season = 1 + 0.4 * np.sin(2 * np.pi * (t % 52) / 52)
        qty = rng.poisson(np.maximum(base * season * (1 + trend * t / 10), 0.5))
        stock = np.zeros(len(weeks)); st = rng.uniform(50, 400)
        for i in range(len(weeks)):
            st = max(st - qty[i] + (rng.uniform(0, 80) if rng.random() < 0.3 else 0), 0); stock[i] = st
        rows.append(pd.DataFrame({'sku_id': f'SKU{s:04d}', 'week_start': weeks, 'qty': qty, 'stock_end': stock,
                                  'category': f'C{s % 6}', 'avg_price': rng.uniform(50, 900)}))
    d = pd.concat(rows, ignore_index=True).sort_values(KEY)
    g = d.groupby('sku_id')
    for l in (1, 2, 4, 8): d[f'qty_lag{l}'] = g['qty'].shift(l)
    for w in (4, 8):
        d[f'qty_rmean{w}'] = g['qty'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        d[f'qty_rstd{w}']  = g['qty'].transform(lambda x: x.rolling(w, min_periods=2).std())
    d['qty_slope4'] = d['qty_rmean4'] - g['qty_rmean4'].shift(4)
    d['y_qty'] = g['qty'].shift(-1)
    d['product_age_weeks'] = g.cumcount()
    d['stock_delta'] = g['stock_end'].diff()
    prev4 = g['qty'].transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
    d['cov_past'] = d['stock_end'] / np.maximum(prev4, EPS)
    fwd = sum(g['qty'].shift(-k) for k in range(1, HORIZON_WEEKS + 1)) / HORIZON_WEEKS
    d['cov_fwd'] = d['stock_end'] / np.maximum(fwd, EPS)
    d['n_fwd_weeks'] = g['qty'].transform(lambda x: x[::-1].expanding().count()[::-1] - 1).clip(upper=HORIZON_WEEKS)
    d['trend4'] = d['qty_slope4']
    sales = d[KEY + ['qty','y_qty','avg_price','category','product_age_weeks'] + [c for c in d.columns if c.startswith('qty_')]]
    stock = d[KEY + ['stock_end','stock_delta','cov_past','cov_fwd','n_fwd_weeks','trend4']]
    return sales.copy(), stock.copy()

if USE_SYNTHETIC:
    _s, _k = make_synthetic_inputs(); save_df(_s, 'es_sales_wk'); save_df(_k, 'es_stock_wk')
    print('⚠️ Đang dùng DỮ LIỆU GIẢ (USE_SYNTHETIC=True) – chỉ để thử khung!')

In [ ]:
# ---- 4.1 Nạp kết quả NB1, NB2 + kiểm tra hợp đồng file ✅ ----
es_sales_wk = load_df('es_sales_wk')
es_stock_wk = load_df('es_stock_wk')
HAVE_INPUTS = es_sales_wk is not None and es_stock_wk is not None
if not HAVE_INPUTS:
    print('❌ Chưa thấy es_sales_wk / es_stock_wk trong', DIRS['interim'],
          '\n   → chạy NB1, NB2 (Mục 1) rồi copy file xuất vào interim/, hoặc bật USE_SYNTHETIC để thử khung.')
else:
    validate_schema(es_sales_wk, 'es_sales_wk'); validate_schema(es_stock_wk, 'es_stock_wk')
    describe_df(es_sales_wk, 'es_sales_wk (NB1)'); describe_df(es_stock_wk, 'es_stock_wk (NB2)')

In [ ]:
# ---- 4.2 🎬 DEMO-1: nạp bảng chất lượng/danh mục do NB1, NB2 xuất (nếu có) ----
es_table_catalog     = load_df('es_table_catalog')
es_quality_sales     = load_df('es_quality_sales')
es_quality_inventory = load_df('es_quality_inventory')
es_filter_log        = load_df('es_filter_log')
ic_feasibility       = load_df('ic_feasibility')
for nm, d_ in [('Danh mục bảng', es_table_catalog), ('Chất lượng – bán hàng', es_quality_sales),
               ('Chất lượng – tồn kho', es_quality_inventory), ('Log lọc NB1', es_filter_log), ('Instacart – khả thi', ic_feasibility)]:
    print(f'\n### {nm}'); display(d_.head(15)) if d_ is not None else print('   (chưa có file)')

In [ ]:
# ---- 4.3 🎬 DEMO-1: tổng quan trực quan từ kết quả NB1/NB2 ----
import matplotlib.pyplot as plt
if HAVE_INPUTS:
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    es_sales_wk.groupby('week_start')['qty'].sum().plot(ax=ax[0]); ax[0].set_title('Tổng doanh số theo tuần (xu hướng, mùa vụ)')
    es_sales_wk['y_qty'].dropna().clip(upper=es_sales_wk['y_qty'].quantile(0.99)).plot.hist(bins=50, ax=ax[1], logy=True); ax[1].set_title('Phân bố y_qty (cắt P99, trục log)')
    es_stock_wk['cov_fwd'].replace([np.inf,-np.inf],np.nan).dropna().clip(upper=es_stock_wk['cov_fwd'].quantile(0.99)).plot.hist(bins=50, ax=ax[2]); ax[2].set_title('Phân bố cov_fwd (chưa gán lớp)')
    plt.tight_layout(); plt.show(); save_fig(fig, 'tong_quan_khao_sat')

## 5. [A3] Lọc – nối – chia dữ liệu theo thời gian  🎬 DEMO-2

**Người phụ trách:** 🟦 ___ · **Cần kết quả của:** A2, A11 (NB1, NB2) · **Cho:** A4, A5, A6

| Việc nhỏ | Nội dung |
|---|---|
| 5.1 | Nối bán hàng + tồn kho trên `(sku_id, week_start)` |
| 5.2 | Lọc cửa sổ chung, bỏ dòng thiếu nhãn/không đủ 4 tuần nhìn trước |
| 5.3 | Bảng số mẫu sau từng bước lọc |
| 5.4 | Chia Train \| gap \| Val \| gap \| Test theo **mốc tuần chung** |
| 5.5 | Lọc SKU theo lịch sử **trong train** (`MIN_HISTORY_WEEKS`) |
| 5.6 | Ngưỡng P60/P85 **chỉ từ train** → gán Low/Medium/High |
| 5.7 | ✅ Kiểm tra rò rỉ |
| 5.8 | Lưu `mt_train/val/test` |

In [ ]:
# ---- 5.1–5.3 Nối, lọc, bảng số mẫu ----
assert HAVE_INPUTS, 'Thiếu input – xem Mục 4'
STEP_LOG.clear()
mt = es_sales_wk.merge(es_stock_wk, on=KEY, how='inner', suffixes=('', '_stk'))
log_step(mt, '5.1 nối bán hàng + tồn kho')

mt = mt[(mt.week_start >= SYNC_START) & (mt.week_start <= SYNC_END)]
log_step(mt, '5.2a trong cửa sổ chung')
mt = mt[mt['n_fwd_weeks'] >= HORIZON_WEEKS]
log_step(mt, '5.2b đủ 4 tuần nhìn trước')
mt = mt.dropna(subset=['y_qty', 'cov_fwd', 'stock_end'])
mt = mt[np.isfinite(mt['cov_fwd'])]
log_step(mt, '5.2c bỏ dòng thiếu nhãn')

mt['week_of_year'] = mt.week_start.dt.isocalendar().week.astype(int); mt['month'] = mt.week_start.dt.month
if 'trend4' not in mt.columns: mt['trend4'] = mt.get('qty_slope4', 0)
mt['trend4'] = mt['trend4'].fillna(0)
pd.DataFrame(STEP_LOG)

In [ ]:
# ---- 5.4 Chia theo thời gian (mốc tuần chung) 🎬 ----
def make_time_split(weeks, ratios=SPLIT_RATIOS, gap=GAP_WEEKS):
    w = np.array(sorted(pd.unique(weeks))); n = len(w); usable = n - 2 * gap
    n_tr = int(round(usable * ratios[0])); n_va = int(round(usable * ratios[1]))
    tr = w[:n_tr]; va = w[n_tr + gap: n_tr + gap + n_va]; te = w[n_tr + gap + n_va + gap:]
    return tr, va, te

wk_tr, wk_va, wk_te = make_time_split(mt.week_start)
mt['split'] = np.select([mt.week_start.isin(wk_tr), mt.week_start.isin(wk_va), mt.week_start.isin(wk_te)], ['train','val','test'], default='gap')
print(mt['split'].value_counts().to_string())
split_table = pd.DataFrame({'tập': ['train','gap','val','gap','test'],
    'tuần_đầu': [wk_tr[0], None, wk_va[0], None, wk_te[0]], 'tuần_cuối': [wk_tr[-1], None, wk_va[-1], None, wk_te[-1]],
    'số_tuần': [len(wk_tr), GAP_WEEKS, len(wk_va), GAP_WEEKS, len(wk_te)]})
display(split_table)
mt = mt[mt['split'] != 'gap'].copy(); log_step(mt, '5.4 bỏ các tuần gap')

In [ ]:
# ---- 5.5 Lọc SKU theo lịch sử trong TRAIN (chống cold-start) ----
hist = mt[mt.split == 'train'].groupby('sku_id').week_start.nunique()
keep_sku = hist[hist >= MIN_HISTORY_WEEKS].index
print(f'SKU đủ lịch sử train (>= {MIN_HISTORY_WEEKS} tuần): {len(keep_sku)} / {mt.sku_id.nunique()}')
mt = mt[mt.sku_id.isin(keep_sku)].copy(); log_step(mt, '5.5 SKU đủ lịch sử train')

# mã hóa biến phân loại: mapping CHỈ từ train, giá trị lạ -> 0
CAT_USED = [c for c in CAT_COLS if c in mt.columns]
CAT_MAP = {}
for c in CAT_USED:
    cats = sorted(mt.loc[mt.split == 'train', c].astype(str).unique()); CAT_MAP[c] = {v: i + 1 for i, v in enumerate(cats)}
    mt[c + '_id'] = mt[c].astype(str).map(CAT_MAP[c]).fillna(0).astype(int)
print('Cột phân loại dùng:', CAT_USED)

In [ ]:
# ---- 5.6 Ngưỡng nhãn CHỈ TỪ TRAIN → gán Low/Medium/High 🎬 ----
def fit_risk_thresholds(train, p_med=RISK_PCT_MED, p_high=RISK_PCT_HIGH):
    return {'T_med': float(np.percentile(train['cov_fwd'], p_med)), 'T_high': float(np.percentile(train['cov_fwd'], p_high))}
def assign_risk(df, th):
    high = (df['cov_fwd'] >= th['T_high']) & (df['trend4'] <= 0)             # tồn kho cao + nhu cầu không tăng
    med  = (df['cov_fwd'] >= th['T_med']) & ~high
    return np.where(high, 2, np.where(med, 1, 0))

RISK_TH = fit_risk_thresholds(mt[mt.split == 'train']); print('Ngưỡng từ train:', RISK_TH)
mt['risk'] = assign_risk(mt, RISK_TH); mt['risk_name'] = mt['risk'].map(RISK_NAMES)
display(pd.crosstab(mt['split'], mt['risk_name'], normalize='index').round(3).loc[['train','val','test']])
mt_train, mt_val, mt_test = [mt[mt.split == s].copy() for s in ('train','val','test')]

In [ ]:
# ---- 5.7 ✅ KIỂM TRA RÒ RỈ (không được bỏ) ----
FEATURES = [c for g in FEATURE_GROUPS.values() for c in g if c in mt.columns]
CAT_IDS  = [c + '_id' for c in CAT_USED]
assert not set(FEATURES) & set(LEAK_COLS), 'Đặc trưng chứa cột nhãn!'
assert mt_train.week_start.max() < mt_val.week_start.min() < mt_val.week_start.max() < mt_test.week_start.min(), 'Các tập chồng thời gian'
gap_tv = (mt_val.week_start.min() - mt_train.week_start.max()).days // 7
gap_vt = (mt_test.week_start.min() - mt_val.week_start.max()).days // 7
assert (gap_tv - 1) >= GAP_WEEKS and (gap_vt - 1) >= GAP_WEEKS, 'Gap < 4 tuần'   # số tuần bị bỏ giữa hai tập
assert set(mt_val.sku_id) <= set(mt_train.sku_id) and set(mt_test.sku_id) <= set(mt_train.sku_id), 'SKU lạ ở val/test'
for nm, d_ in [('train', mt_train), ('val', mt_val), ('test', mt_test)]: assert d_.duplicated(KEY).sum() == 0, f'{nm} trùng khóa'
print('✅ Không rò rỉ: đặc trưng ∩ nhãn = ∅ | tập không chồng, gap tuần:', gap_tv, gap_vt, '| SKU val/test ⊆ train | khóa duy nhất')
print('Đặc trưng số dùng:', FEATURES); print('Cột phân loại dùng:', CAT_IDS)

In [ ]:
# ---- 5.8 Lưu ba tập + ngưỡng (dùng chung cho A4, A5, A6, A8) ----
for nm, d_ in [('mt_train', mt_train), ('mt_val', mt_val), ('mt_test', mt_test)]: save_df(d_, nm, 'processed')
json.dump({'RISK_TH': RISK_TH, 'FEATURES': FEATURES, 'CAT_IDS': CAT_IDS, 'SYNC': [SYNC_START, SYNC_END],
           'weeks': {'train': [str(wk_tr[0]), str(wk_tr[-1])], 'val': [str(wk_va[0]), str(wk_va[-1])], 'test': [str(wk_te[0]), str(wk_te[-1])]}},
          open(DIRS['processed'] / 'split_meta.json', 'w'), ensure_ascii=False, indent=2)
pd.DataFrame(STEP_LOG).to_csv(DIRS['results'] / 'sample_count_log.csv', index=False)
print('đã lưu mt_train/val/test + split_meta.json')

## 6. [A6] Phân bố biến mục tiêu  🎬 DEMO-1

**Người phụ trách:** 🟦 ___ · **Cần:** A3 · **Cho:** A7 (DataInventory), Thuyết minh Mục 5

| Việc nhỏ | Nội dung |
|---|---|
| 6.1 | Phân bố `y_qty` theo train/val/test (mean, median, std, phân vị, tỷ lệ = 0) |
| 6.2 | Số mẫu và tỷ lệ Low/Medium/High từng tập + theo nhóm sản phẩm |
| 6.3 | Kiểm tra ngưỡng P60/P85 và tỷ lệ High (so với 12,2% ở kiểm chứng sơ bộ) |
| 6.4 | Biểu đồ phân bố `cov_fwd` có vạch ngưỡng |

In [ ]:
# ---- 6.1 – 6.3 ----
def describe_target(df, col='y_qty'):
    q = df[col].quantile([.25, .5, .75, .9, .99]).rename({.25:'P25',.5:'P50',.75:'P75',.9:'P90',.99:'P99'})
    return pd.concat([pd.Series({'n': len(df), 'mean': df[col].mean(), 'std': df[col].std(), 'min': df[col].min(), 'max': df[col].max(),
                                 'tỷ_lệ_bằng_0': (df[col] == 0).mean()}), q])
tbl_target = pd.DataFrame({s: describe_target(d_) for s, d_ in [('train', mt_train), ('val', mt_val), ('test', mt_test)]}).round(3)
display(tbl_target)
tbl_risk_n = pd.crosstab(mt['split'], mt['risk_name']).loc[['train','val','test']]
tbl_risk_p = pd.crosstab(mt['split'], mt['risk_name'], normalize='index').loc[['train','val','test']].round(4)
display(tbl_risk_n); display(tbl_risk_p)
print('Tỷ lệ High (train):', round(float((mt_train.risk == 2).mean()), 4), '(tham chiếu sơ bộ: 0.122)')
if 'category' in mt.columns: display(pd.crosstab(mt['category'], mt['risk_name'], normalize='index').round(3))
tbl_target.to_csv(DIRS['results'] / 'target_dist_y_qty.csv'); tbl_risk_n.to_csv(DIRS['results'] / 'target_dist_risk.csv')

In [ ]:
# ---- 6.4 Biểu đồ cov_fwd có vạch ngưỡng ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
x = mt_train['cov_fwd'].clip(upper=mt_train['cov_fwd'].quantile(0.99))
ax[0].hist(x, bins=60); ax[0].axvline(RISK_TH['T_med'], c='orange', ls='--', label=f"P{RISK_PCT_MED}"); ax[0].axvline(RISK_TH['T_high'], c='red', ls='--', label=f"P{RISK_PCT_HIGH}")
ax[0].set_title('cov_fwd (train) và ngưỡng nhãn'); ax[0].legend()
tbl_risk_p.plot.bar(stacked=True, ax=ax[1]); ax[1].set_title('Tỷ lệ Low/Medium/High theo tập'); plt.tight_layout(); plt.show(); save_fig(fig, 'phan_bo_nhan_rui_ro')

## 7. [A4] Baseline cho cả hai nhiệm vụ

**Người phụ trách:** 🟦 ___ · **Cần:** A3 · **Cho:** A5, A8, A9

Quy tắc: cùng tập chia, cùng đặc trưng; siêu tham số chọn trên **validation**; **test chạy một lần**; báo cáo 3–5 seed nếu mô hình ngẫu nhiên.

| Mã | Baseline | Mục |
|---|---|---|
| S0 / S1 / S3 / S4 | Naive · Moving average (4, 8 tuần) · Ridge · RF/LightGBM (doanh số) | 7.2 |
| R0 / R1 / R2 / R3 | Lớp đa số · luật ngưỡng hiện hành · Logistic · cây tăng cường (rủi ro) | 7.3 |
| P1 | **Pipeline hai bước**: dự báo → `stock_end / ŷ` → ngưỡng | 7.4 |
| Ablation nhanh | Bỏ `cov_past`, `stock_end` khỏi R2/R3 (kiểm tra điểm cao có chỉ do luật tồn kho?) | 7.5 |

In [ ]:
# ---- 7.1 Hàm đánh giá + chuẩn bị ma trận ----
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, precision_recall_fscore_support,
                             f1_score, confusion_matrix, average_precision_score, roc_auc_score)
try:
    import lightgbm as lgb; HAVE_LGB = True
except Exception: HAVE_LGB = False

def eval_reg(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float); m = y > 0
    return {'MAE': mean_absolute_error(y, yhat), 'RMSE': math.sqrt(mean_squared_error(y, yhat)),
            'WAPE': float(np.abs(y - yhat).sum() / max(np.abs(y).sum(), EPS)), 'R2': r2_score(y, yhat),
            'MAPE(qty>0)': float(np.mean(np.abs((y[m] - yhat[m]) / y[m]))) if m.any() else np.nan}

def eval_clf(y, pred, proba=None):
    y = np.asarray(y); pred = np.asarray(pred)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    out = {'Recall_High': r[2], 'Precision_High': p[2], 'F1_High': f[2], 'Macro_F1': f1_score(y, pred, average='macro', labels=[0,1,2], zero_division=0)}
    if proba is not None:
        try: out['PR_AUC_High'] = average_precision_score((y == 2).astype(int), proba[:, 2])
        except Exception: out['PR_AUC_High'] = np.nan
        try: out['AUC_ROC_ovr'] = roc_auc_score(y, proba, multi_class='ovr', labels=[0, 1, 2])
        except Exception: out['AUC_ROC_ovr'] = np.nan
    return out

# giá trị điền thiếu + scaler: CHỈ fit trên train
FEATS = FEATURES + CAT_IDS
MED = mt_train[FEATURES].median()
def X_of(df, feats=None):
    f = feats or FEATS; return df[f].copy().fillna(MED.reindex(f).fillna(0))
sc = StandardScaler().fit(X_of(mt_train))
Xtr, Xva, Xte = [X_of(d_) for d_ in (mt_train, mt_val, mt_test)]
ytr_q, yva_q, yte_q = mt_train.y_qty.values, mt_val.y_qty.values, mt_test.y_qty.values
ytr_r, yva_r, yte_r = mt_train.risk.values, mt_val.risk.values, mt_test.risk.values
res_reg, res_clf, PRED = [], [], {}      # res_* : bảng kết quả; PRED : dự đoán lưu lại cho A8
def add_res(lst, name, split, metrics): lst.append({'model': name, 'split': split, **metrics})

In [ ]:
# ---- 7.2 Dự báo doanh số: S0, S1, S3, S4 ----
def reg_row(name, preds):                  # preds: dict split -> yhat
    for s, y in (('val', yva_q), ('test', yte_q)): add_res(res_reg, name, s, eval_reg(y, preds[s]))
    PRED[name] = preds

PRED_S = {}
reg_row('S0_naive', {'val': mt_val.qty.values, 'test': mt_test.qty.values})
for w in (4, 8):
    c = f'qty_rmean{w}'
    if c in mt.columns: reg_row(f'S1_ma{w}', {'val': mt_val[c].fillna(mt_val.qty).values, 'test': mt_test[c].fillna(mt_test.qty).values})
ridge = Ridge(alpha=1.0).fit(sc.transform(Xtr), ytr_q)
reg_row('S3_ridge', {'val': np.clip(ridge.predict(sc.transform(Xva)), 0, None), 'test': np.clip(ridge.predict(sc.transform(Xte)), 0, None)})
if HAVE_LGB:
    m4 = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbose=-1)
    m4.fit(Xtr, ytr_q, eval_set=[(Xva, yva_q)], callbacks=[lgb.early_stopping(30, verbose=False)])
    name4 = 'S4_lightgbm'
else:
    m4 = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED).fit(Xtr, ytr_q); name4 = 'S4_rf'
reg_row(name4, {'val': np.clip(m4.predict(Xva), 0, None), 'test': np.clip(m4.predict(Xte), 0, None)})
res_reg_df = pd.DataFrame(res_reg); display(res_reg_df[res_reg_df.split == 'val'].round(4))

In [ ]:
# ---- 7.3 Rủi ro: R0, R1, R2, R3 ----
def proba_full(model, X):                  # đảm bảo đủ 3 cột lớp theo thứ tự 0,1,2
    p = model.predict_proba(X); out = np.zeros((len(X), 3))
    for i, c in enumerate(model.classes_): out[:, int(c)] = p[:, i]
    return out
def clf_row(name, preds, probas=None):
    for s, y in (('val', yva_r), ('test', yte_r)): add_res(res_clf, name, s, eval_clf(y, preds[s], None if probas is None else probas[s]))
    PRED[name] = {'pred': preds, 'proba': probas}

clf_row('R0_majority', {'val': np.zeros(len(yva_r), int), 'test': np.zeros(len(yte_r), int)})
# R1: luật ngưỡng hiện hành trên cov_past; 🟦 TODO: có thể thay bằng ngưỡng nghiệp vụ cố định (ví dụ "tồn >= X tuần bán") và GHI RÕ trước khi xem test
TH_RULE = {'med': float(mt_train['cov_past'].quantile(RISK_PCT_MED / 100)), 'high': float(mt_train['cov_past'].quantile(RISK_PCT_HIGH / 100))}   # bỏ qua NaN
rule = lambda d_: np.where(d_['cov_past'].fillna(0) >= TH_RULE['high'], 2, np.where(d_['cov_past'].fillna(0) >= TH_RULE['med'], 1, 0))
print('Ngưỡng luật R1 (từ train):', TH_RULE)
clf_row('R1_rule_cov_past', {'val': rule(mt_val), 'test': rule(mt_test)})
logit = LogisticRegression(max_iter=2000, class_weight='balanced').fit(sc.transform(Xtr), ytr_r)
clf_row('R2_logistic', {s: logit.predict(sc.transform(X_)) for s, X_ in (('val', Xva), ('test', Xte))},
        {s: proba_full(logit, sc.transform(X_)) for s, X_ in (('val', Xva), ('test', Xte))})
if HAVE_LGB:
    m3 = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, class_weight='balanced', random_state=SEED, verbose=-1)
    m3.fit(Xtr, ytr_r, eval_set=[(Xva, yva_r)], callbacks=[lgb.early_stopping(30, verbose=False)]); name3 = 'R3_lightgbm'
else:
    m3 = RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=SEED).fit(Xtr, ytr_r); name3 = 'R3_rf'
clf_row(name3, {s: m3.predict(X_) for s, X_ in (('val', Xva), ('test', Xte))}, {s: proba_full(m3, X_) for s, X_ in (('val', Xva), ('test', Xte))})
res_clf_df = pd.DataFrame(res_clf); display(res_clf_df[res_clf_df.split == 'val'].round(4))

In [ ]:
# ---- 7.4 P1: pipeline hai bước (dự báo → cov_pred = stock_end/ŷ → ngưỡng chọn trên VALIDATION) ----
best_s = res_reg_df[(res_reg_df.split == 'val') & (res_reg_df.model.str.startswith(('S3', 'S4')))].sort_values('MAE').iloc[0].model
print('Mô hình dự báo dùng cho P1 (chọn theo MAE val):', best_s)
def p1_classify(d_, yhat, tm, th):
    cov = d_['stock_end'].values / np.maximum(yhat, EPS); return np.where(cov >= th, 2, np.where(cov >= tm, 1, 0)), cov
cov_val = mt_val['stock_end'].values / np.maximum(PRED[best_s]['val'], EPS)
best = (-1, None)
for pm in (50, 55, 60, 65, 70):
    for ph in (75, 80, 85, 90, 95):
        tm, th = np.percentile(cov_val, [pm, ph]); pr, _ = p1_classify(mt_val, PRED[best_s]['val'], tm, th)
        f = f1_score(yva_r, pr, average='macro', labels=[0,1,2], zero_division=0)
        if f > best[0]: best = (f, (tm, th, pm, ph))
P1_TH = best[1]; print('Ngưỡng P1 chọn trên val (tm, th, pm, ph):', P1_TH)
clf_row('P1_pipeline_2buoc', {s: p1_classify(d_, PRED[best_s][s], P1_TH[0], P1_TH[1])[0] for s, d_ in (('val', mt_val), ('test', mt_test))})
res_clf_df = pd.DataFrame(res_clf); display(res_clf_df.round(4))

In [ ]:
# ---- 7.5 Ablation nhanh: bỏ cov_past, stock_end khỏi R3 (cảnh báo "nhãn gần với hàm của đặc trưng") ----
if HAVE_LGB:
    f_abl = [f for f in FEATS if f not in ('cov_past', 'stock_end')]
    m_abl = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, class_weight='balanced', random_state=SEED, verbose=-1)
    m_abl.fit(X_of(mt_train, f_abl), ytr_r, eval_set=[(X_of(mt_val, f_abl), yva_r)], callbacks=[lgb.early_stopping(30, verbose=False)])
    pa = proba_full(m_abl, X_of(mt_val, f_abl)); add_res(res_clf, 'R3_bo_cov_past_stock', 'val', eval_clf(yva_r, pa.argmax(1), pa))
    display(pd.DataFrame(res_clf).query("split=='val'").round(4))
    print('→ Nếu điểm R3 gần như không đổi khi bỏ cov_past/stock_end: mô hình học tín hiệu khác luật tồn kho.')

## 8. [A5] Transformer đơn nhiệm và đa nhiệm  ⏱ NẶNG

**Người phụ trách:** 🟦 ___ · **Cần:** A3, A4 · **Cho:** A8, A9, Thuyết minh Mục 7, 12

| Việc nhỏ | Nội dung |
|---|---|
| 8.1 | Kiến trúc: feature-token (kiểu TabTransformer) → encoder chung → Sales Head + Risk Head |
| 8.2 | Huấn luyện `mode ∈ {sales, risk, multi}`; loss `λ1·SmoothL1(log1p qty) + λ2·CE(class weight)` |
| 8.3 | Chạy S6 (chỉ doanh số), R4 (chỉ rủi ro), MT (đa nhiệm) — 3–5 seed |
| 8.4 | Chọn λ trên val; ablation nhóm đặc trưng |
| 8.5 | So sánh với baseline và P1 |

> Cờ `RUN_HEAVY=False` → các cell dưới sẽ in lời nhắc và bỏ qua. Cần PyTorch (Colab đã có sẵn).

In [ ]:
# ---- 8.1 Kiến trúc (khung) ----
try:
    import torch, torch.nn as nn
    HAVE_TORCH = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    HAVE_TORCH = False; print('⚠️ Chưa có PyTorch – bỏ qua Mục 8')

if HAVE_TORCH:
    class FeatureTokenizer(nn.Module):
        def __init__(self, n_num, d, cat_sizes):
            super().__init__()
            self.w = nn.Parameter(torch.randn(n_num, d) * 0.02); self.b = nn.Parameter(torch.zeros(n_num, d))
            self.cat_emb = nn.ModuleList([nn.Embedding(s + 1, d) for s in cat_sizes])
        def forward(self, xn, xc):
            tok = xn.unsqueeze(-1) * self.w + self.b                                # B × n_num × d
            if len(self.cat_emb):
                tok = torch.cat([tok] + [e(xc[:, i]).unsqueeze(1) for i, e in enumerate(self.cat_emb)], 1)
            return tok

    class MTTransformer(nn.Module):
        def __init__(self, n_num, cat_sizes, d=32, heads=4, layers=2, ff=64, drop=0.1, n_cls=3):
            super().__init__()
            self.tok = FeatureTokenizer(n_num, d, cat_sizes); self.cls = nn.Parameter(torch.zeros(1, 1, d))
            enc = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True); self.enc = nn.TransformerEncoder(enc, layers)
            self.head_s = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.ReLU(), nn.Linear(d, 1))      # Sales Head
            self.head_r = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.ReLU(), nn.Linear(d, n_cls))  # Risk Head
        def forward(self, xn, xc):
            t = self.tok(xn, xc); t = torch.cat([self.cls.expand(len(t), -1, -1), t], 1)
            h = self.enc(t)[:, 0]; return self.head_s(h).squeeze(-1), self.head_r(h)

In [ ]:
# ---- 8.2 Hàm huấn luyện/dự đoán (khung) ----
if HAVE_TORCH:
    NUM_F = FEATURES; CAT_F = CAT_IDS; CAT_SIZES = [len(CAT_MAP[c.replace('_id', '')]) for c in CAT_F]
    MU = mt_train[NUM_F].fillna(MED).mean(); SD = mt_train[NUM_F].fillna(MED).std().replace(0, 1)     # fit trên train
    def tensors(df):
        xn = ((df[NUM_F].fillna(MED) - MU) / SD).values.astype('float32')
        xc = df[CAT_F].values.astype('int64') if CAT_F else np.zeros((len(df), 0), 'int64')
        return torch.tensor(xn), torch.tensor(xc), torch.tensor(np.log1p(df.y_qty.values).astype('float32')), torch.tensor(df.risk.values.astype('int64'))

    def train_mt(mode='multi', lam=(1.0, 1.0), epochs=60, lr=1e-3, bs=512, patience=8, seed=SEED, d=32, layers=2):
        set_seed(seed)
        xn, xc, ys, yr = tensors(mt_train); vn, vc, vs, vr = tensors(mt_val)
        model = MTTransformer(len(NUM_F), CAT_SIZES, d=d, layers=layers).to(DEVICE)
        cw = torch.tensor(len(ytr_r) / (3 * np.maximum(np.bincount(ytr_r, minlength=3), 1)), dtype=torch.float32).to(DEVICE)
        ls_fn, lr_fn = nn.SmoothL1Loss(), nn.CrossEntropyLoss(weight=cw)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        def loss_of(ps, pr, ys_, yr_):
            a = lam[0] * ls_fn(ps, ys_) if mode in ('multi', 'sales') else 0.0
            b = lam[1] * lr_fn(pr, yr_)  if mode in ('multi', 'risk')  else 0.0
            return a + b, a, b
        best, bad, state, hist = 1e9, 0, None, []
        for ep in range(epochs):
            model.train(); idx = torch.randperm(len(xn))
            for i in range(0, len(idx), bs):
                j = idx[i:i + bs]; ps, pr = model(xn[j].to(DEVICE), xc[j].to(DEVICE))
                l, _, _ = loss_of(ps, pr, ys[j].to(DEVICE), yr[j].to(DEVICE)); opt.zero_grad(); l.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                ps, pr = model(vn.to(DEVICE), vc.to(DEVICE)); l, a, b = loss_of(ps, pr, vs.to(DEVICE), vr.to(DEVICE))
            hist.append({'epoch': ep, 'val_total': float(l), 'val_sales': float(a), 'val_risk': float(b)})
            if float(l) < best - 1e-4: best, bad, state = float(l), 0, {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience: break
        model.load_state_dict(state); return model, pd.DataFrame(hist)

    def predict_mt(model, df):
        xn, xc, _, _ = tensors(df); model.eval()
        with torch.no_grad(): ps, pr = model(xn.to(DEVICE), xc.to(DEVICE))
        return np.clip(np.expm1(ps.cpu().numpy()), 0, None), torch.softmax(pr, -1).cpu().numpy()

In [ ]:
# ---- 8.3 ⏱ NẶNG: chạy S6, R4, MT (3 seed) ----
SEEDS = [42, 43, 44]
if HAVE_TORCH and RUN_HEAVY:
    for mode, tag in (('sales', 'S6_tf_sales'), ('risk', 'R4_tf_risk'), ('multi', 'MT_tf_multi')):
        for sd in SEEDS:
            model, h = train_mt(mode, lam=(1.0, 1.0), seed=sd)
            _y, _p = {}, {}
            for s, d_, y_q, y_r in (('val', mt_val, yva_q, yva_r), ('test', mt_test, yte_q, yte_r)):
                yh, pb = predict_mt(model, d_); _y[s], _p[s] = yh, pb
                if mode in ('sales', 'multi'): add_res(res_reg, f'{tag}|seed{sd}', s, eval_reg(y_q, yh))
                if mode in ('risk', 'multi'):  add_res(res_clf, f'{tag}|seed{sd}', s, eval_clf(y_r, pb.argmax(1), pb))
            if sd == SEEDS[0]:        # lưu dự đoán của seed đầu để dùng ở Mục 9 (MODEL_FINAL = 'MT_tf_multi')
                if mode == 'sales': PRED[tag] = _y
                else: PRED[tag] = {'pred': {k: v.argmax(1) for k, v in _p.items()}, 'proba': _p}
                if mode == 'multi': PRED[tag + '_sales'] = _y
            print(tag, 'seed', sd, 'xong, epoch dùng:', len(h))
else:
    print('⏱ Bỏ qua (cần RUN_HEAVY=True và PyTorch).')

In [ ]:
# ---- 8.4 ⏱ NẶNG: chọn λ trên val + ablation nhóm đặc trưng (khung) ----
LAMBDAS = [(1.0, 0.5), (1.0, 1.0), (1.0, 2.0)]        # 🟦 TODO: điều chỉnh lưới
if HAVE_TORCH and RUN_HEAVY:
    rows = []
    for lam in LAMBDAS:
        model, h = train_mt('multi', lam=lam, seed=SEED); yh, pb = predict_mt(model, mt_val)
        rows.append({'lam_sales': lam[0], 'lam_risk': lam[1], **{'val_' + k: v for k, v in eval_reg(yva_q, yh).items() if k in ('MAE', 'WAPE')},
                     **{'val_' + k: v for k, v in eval_clf(yva_r, pb.argmax(1), pb).items() if k in ('Recall_High', 'PR_AUC_High', 'Macro_F1')}})
    res_lambda = pd.DataFrame(rows); display(res_lambda.round(4)); res_lambda.to_csv(DIRS['results'] / 'lambda_sensitivity.csv', index=False)
else:
    print('⏱ Bỏ qua.')
# Ablation nhóm đặc trưng (bỏ inventory / price_promo / ...): lặp train_mt với NUM_F đã loại nhóm – 🟦 TODO nếu còn thời gian

In [ ]:
# ---- 8.5 Lưu và so sánh toàn bộ kết quả ----
res_reg_df, res_clf_df = pd.DataFrame(res_reg), pd.DataFrame(res_clf)
res_reg_df.to_csv(DIRS['results'] / 'res_reg.csv', index=False); res_clf_df.to_csv(DIRS['results'] / 'res_clf.csv', index=False)
print('— Dự báo doanh số (test) —'); display(res_reg_df[res_reg_df.split == 'test'].round(4))
print('— Rủi ro tồn kho (test) —'); display(res_clf_df[res_clf_df.split == 'test'].round(4))

## 9. [A8] Độ đo nghiệp vụ và hỗ trợ quyết định  🎬 DEMO-3

**Người phụ trách:** 🟦 ___ · **Cần:** A4, A5 · **Cho:** A9, Thuyết minh Mục 8

| Việc nhỏ | Nội dung |
|---|---|
| 9.1 | Số SKU cần rà soát/tuần, tỷ lệ cảnh báo sai |
| 9.2 | Tồn kho rủi ro cao phát hiện sớm (theo số SKU và theo giá trị), Coverage nhóm cảnh báo |
| 9.3 | 🎬 Bảng **Top-K SKU rủi ro cao** của một tuần (demo hỗ trợ quyết định) |
| 9.4 | So sánh với luật ngưỡng R1 và pipeline P1 |

In [ ]:
# ---- 9.1 – 9.2 Hàm độ đo nghiệp vụ (🟦 TODO: chọn mô hình tốt nhất làm "MODEL_FINAL") ----
def business_metrics(df, pred, yhat=None, value_col=None):
    d_ = df[['sku_id', 'week_start', 'stock_end', 'risk']].copy(); d_['pred'] = np.asarray(pred)
    if value_col and value_col in df.columns: d_['value'] = df['stock_end'] * df[value_col]
    alert = d_[d_.pred == 2]; true_high = d_[d_.risk == 2]
    per_week = alert.groupby('week_start').size().reindex(d_.week_start.unique(), fill_value=0)
    out = {'SKU_rà_soát_TB_mỗi_tuần': per_week.mean(), 'SKU_rà_soát_max_tuần': per_week.max(),
           'tỷ_lệ_cảnh_báo_sai': float((alert.risk != 2).mean()) if len(alert) else np.nan,
           'recall_High_theo_SKU': float((true_high.pred == 2).mean()) if len(true_high) else np.nan}
    if 'value' in d_: out['recall_High_theo_giá_trị'] = float(d_[(d_.risk == 2) & (d_.pred == 2)].value.sum() / max(true_high.value.sum(), EPS))
    if yhat is not None:
        d_['cov_pred'] = d_.stock_end / np.maximum(np.asarray(yhat), EPS)
        out['Coverage_dự_báo_TB_nhóm_cảnh_báo'] = float(d_[d_.pred == 2].cov_pred.mean()); out['Coverage_dự_báo_TB_nhóm_còn_lại'] = float(d_[d_.pred != 2].cov_pred.mean())
    return out

VALUE_COL = 'unit_cost' if 'unit_cost' in mt.columns else None      # 🟦 TODO: cột giá vốn nếu có (để tính theo giá trị)
cands = {'R1_rule': PRED['R1_rule_cov_past']['pred']['test'], 'P1': PRED['P1_pipeline_2buoc']['pred']['test'],
         name3: PRED[name3]['pred']['test']}
tbl_biz = pd.DataFrame({k: business_metrics(mt_test, v, PRED[best_s]['test'], VALUE_COL) for k, v in cands.items()}).T
display(tbl_biz.round(4)); tbl_biz.to_csv(DIRS['results'] / 'business_metrics_test.csv')

In [ ]:
# ---- 9.3 🎬 DEMO-3: Top-K SKU rủi ro cao trong MỘT tuần của test ----
MODEL_FINAL = name3                                   # 🟦 TODO: đổi thành 'MT_tf_multi' khi đã có
TOPK, DEMO_WEEK = 10, mt_test.week_start.iloc[len(mt_test) // 2]
def topk_table(df, proba_high, yhat, week, k=TOPK):
    d_ = df.assign(p_high=proba_high, yhat=yhat, cov_pred=lambda x: x.stock_end / np.maximum(x.yhat, EPS))
    d_ = d_[d_.week_start == week].sort_values('p_high', ascending=False).head(k)
    d_['gợi_ý'] = np.where(d_.p_high >= 0.7, 'Khuyến mãi/giảm giá + hoãn nhập', np.where(d_.p_high >= 0.4, 'Theo dõi sát, giảm lượng nhập', 'Chưa cần xử lý'))
    return d_[['sku_id', 'stock_end', 'yhat', 'cov_pred', 'p_high', 'risk_name', 'gợi_ý']].round(3)
demo_tbl = topk_table(mt_test, PRED[MODEL_FINAL]['proba']['test'][:, 2], PRED[best_s]['test'], DEMO_WEEK)
print('🎬 Tuần demo:', DEMO_WEEK.date()); display(demo_tbl)

## 10. [A9] Ngưỡng chấp nhận bằng số

**Người phụ trách:** 🟦 ___ · **Cần:** A4, A5, A8 · **Cho:** Thuyết minh Mục 8, Mục 2

> **Không điền số khi chưa có kết quả thật.** Điền vào `THRESH` sau khi đã xem baseline/val; kiểm tra “đạt/không đạt” trên test.

In [ ]:
THRESH = {            # 🟦 TODO: None = chưa đặt
    'recall_high_min'          : None,   # Recall High tối thiểu
    'rel_gain_vs_R1_pr_auc'    : None,   # PR-AUC tăng tối thiểu so với R1/P1 (tương đối, ví dụ 0.10 = +10%)
    'false_alert_rate_max'     : None,   # tỷ lệ cảnh báo sai tối đa
    'mae_not_worse_than_S1_pct': None,   # MAE không tệ hơn baseline trung bình trượt quá x%
    'multitask_vs_single_tol'  : None,   # đa nhiệm không suy giảm quá x% so với đơn nhiệm
}
def check_thresholds(model_row, base_row, biz_row):
    rep = []
    def add(name, ok, detail): rep.append({'tiêu_chí': name, 'kết_quả': 'CHƯA ĐẶT NGƯỠNG' if ok is None else ('ĐẠT' if ok else 'KHÔNG ĐẠT'), 'chi_tiết': detail})
    t = THRESH
    add('Recall High', None if t['recall_high_min'] is None else model_row['Recall_High'] >= t['recall_high_min'], f"{model_row['Recall_High']:.3f}")
    add('PR-AUC vs baseline', None if t['rel_gain_vs_R1_pr_auc'] is None else model_row['PR_AUC_High'] >= base_row['PR_AUC_High'] * (1 + t['rel_gain_vs_R1_pr_auc']), f"{model_row['PR_AUC_High']:.3f} vs {base_row['PR_AUC_High']:.3f}")
    add('Cảnh báo sai', None if t['false_alert_rate_max'] is None else biz_row['tỷ_lệ_cảnh_báo_sai'] <= t['false_alert_rate_max'], f"{biz_row['tỷ_lệ_cảnh_báo_sai']:.3f}")
    return pd.DataFrame(rep)
_m = res_clf_df.query("split=='test' and model==@MODEL_FINAL").iloc[0]
_b = res_clf_df.query("split=='test' and model=='R2_logistic'").iloc[0]     # 🟦 baseline so sánh: R1 không có PR-AUC → dùng R2 hoặc P1
display(check_thresholds(_m, _b, tbl_biz.loc[MODEL_FINAL] if MODEL_FINAL in tbl_biz.index else tbl_biz.iloc[0]))

## 11. [A7] Xuất file DataInventory (Excel)

**Người phụ trách:** 🟦 ___ · **Cần:** A1, A2, A6, A11 · **Sản phẩm:** `Lab3_DS317.R11_Nhom5_DataInventory.xlsx`

Các sheet: nguồn dữ liệu · kiểm kê 7 tiêu chí · danh mục bảng · từ điển dữ liệu · chất lượng · biến mục tiêu · riêng tư. Cột **số liệu** tự động; cột **diễn giải/đánh giá** điền tay (🟦).

In [ ]:
def data_dictionary(df, name):
    rows = []
    for c in df.columns:
        s = df[c]; num = pd.api.types.is_numeric_dtype(s)
        rows.append({'bảng': name, 'cột': c, 'kiểu': str(s.dtype), 'tỷ_lệ_thiếu': round(float(s.isna().mean()), 4), 'số_giá_trị_khác_nhau': int(s.nunique()),
                     'min': s.min() if (num or 'datetime' in str(s.dtype)) else None, 'max': s.max() if (num or 'datetime' in str(s.dtype)) else None,
                     'vai_trò': 'nhãn' if c in LEAK_COLS else ('đặc trưng' if c in FEATURES else ('khóa' if c in KEY else '')), 'ý_nghĩa_nghiệp_vụ': ''})   # 🟦 điền tay cột cuối
    return pd.DataFrame(rows)

sheet_nguon = pd.DataFrame([
    {'nguồn': 'Electronics Sales Dataset', 'giấy_phép': 'CC0', 'link': 'https://www.kaggle.com/datasets/fahdelatiby/electronics-sales-dataset-b-tech-style', 'ngày_truy_cập': '🟦', 'cách_thu_thập': 'Tải từ Kaggle', 'mỗi_dòng_là_gì': 'SalesOrderLines: một SKU trong một đơn hàng', 'vai_trò': 'Nguồn chính'},
    {'nguồn': 'Instacart Market Basket Analysis', 'giấy_phép': 'CC0', 'link': 'https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis', 'ngày_truy_cập': '🟦', 'cách_thu_thập': 'Tải từ Kaggle', 'mỗi_dòng_là_gì': 'order_products: một sản phẩm trong một đơn', 'vai_trò': 'Chỉ khảo sát'}])
sheet_kiemke = pd.DataFrame({'tiêu_chí': ['Nguồn gốc', 'Quy mô', 'Cấu trúc', 'Nhãn', 'Chất lượng', 'Thời gian', 'Tính riêng tư'], 'kết_quả': '🟦', 'bằng_chứng': '🟦', 'đánh_giá (Đạt/Cần lưu ý/Không đạt)': '🟦'})
sheet_riengtu = pd.DataFrame({'cột_nhạy_cảm': ['CustomerName'], 'mức_độ': ['Định danh trực tiếp'], 'cách_xử_lý': ['Không dùng; bỏ cột trước khi xuất/công bố']})
dict_all = pd.concat([data_dictionary(es_sales_wk, 'es_sales_wk'), data_dictionary(es_stock_wk, 'es_stock_wk'), data_dictionary(mt, 'mt (bảng mô hình)')], ignore_index=True)
XLSX = DIRS['excel'] / 'Lab3_DS317.R11_Nhom5_DataInventory.xlsx'
with pd.ExcelWriter(XLSX, engine='openpyxl') as xw:
    sheet_nguon.to_excel(xw, sheet_name='1_nguon_du_lieu', index=False); sheet_kiemke.to_excel(xw, sheet_name='2_kiem_ke_7_tieu_chi', index=False)
    (es_table_catalog if es_table_catalog is not None else pd.DataFrame({'ghi_chú': ['🟦 chưa có es_table_catalog từ NB1']})).to_excel(xw, sheet_name='3_danh_muc_bang', index=False)
    dict_all.to_excel(xw, sheet_name='4_tu_dien_du_lieu', index=False)
    pd.concat([x for x in (es_quality_sales, es_quality_inventory) if x is not None] or [pd.DataFrame({'ghi_chú': ['🟦 chưa có bảng chất lượng từ NB1/NB2']})]).to_excel(xw, sheet_name='5_chat_luong', index=False)
    tbl_target.reset_index().to_excel(xw, sheet_name='6a_bien_muc_tieu_y_qty', index=False); tbl_risk_n.reset_index().to_excel(xw, sheet_name='6b_bien_muc_tieu_risk', index=False)
    sheet_riengtu.to_excel(xw, sheet_name='7_rieng_tu', index=False)
print('đã xuất', XLSX)

## 12. [A10] Biểu đồ cho Thuyết minh và Canva

**Người phụ trách:** 🟦 ___ · **Cần:** A2, A6, A11 · Chọn **3–5 hình** cho Mục 5 và viết 2–3 câu nhận xét cho mỗi hình.

| Hình | Dùng ở | Nguồn |
|---|---|---|
| `fig_tong_quan_khao_sat` | Mục 5, slide 5 | Mục 4.3 |
| `fig_phan_bo_nhan_rui_ro` | Mục 5/8, slide 8 | Mục 6.4 |
| `fig_so_do_chia_du_lieu` | Mục 8, slide 8 | Mục 12.1 |
| `fig_so_sanh_mo_hinh` | Mục 7/8, slide 10, 12 | Mục 12.2 |
| 🟦 (hình khác từ NB1/NB2: mùa vụ, thời gian phủ các bảng…) | Mục 5 | NB1/NB2 |

In [ ]:
# ---- 12.1 Sơ đồ chia dữ liệu theo trục thời gian ----
fig, ax = plt.subplots(figsize=(13, 2.2))
for nm, wk, col in (('Train', wk_tr, '#4C78A8'), ('Val', wk_va, '#F58518'), ('Test', wk_te, '#54A24B')):
    ax.barh(0, (pd.Timestamp(wk[-1]) - pd.Timestamp(wk[0])).days, left=pd.Timestamp(wk[0]).toordinal(), color=col, edgecolor='k', label=f'{nm} ({len(wk)} tuần)')
ax.set_yticks([]); ax.set_xlabel('Thời gian (ngày thứ tự)'); ax.set_title('Chia theo thời gian: Train | gap 4 tuần | Val | gap 4 tuần | Test'); ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.45))
plt.show(); save_fig(fig, 'so_do_chia_du_lieu')

In [ ]:
# ---- 12.2 So sánh mô hình (test) ----
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
r = res_reg_df[res_reg_df.split == 'test'].groupby('model')['MAE'].mean().sort_values(); r.plot.barh(ax=ax[0]); ax[0].set_title('Dự báo doanh số – MAE (test, thấp hơn là tốt)')
c = res_clf_df[res_clf_df.split == 'test'].groupby('model')['Recall_High'].mean().sort_values(); c.plot.barh(ax=ax[1], color='#E45756'); ax[1].set_title('Rủi ro tồn kho – Recall High (test, cao hơn là tốt)')
plt.tight_layout(); plt.show(); save_fig(fig, 'so_sanh_mo_hinh')

## 13. [A11] Tổng kết, kiểm tra cuối và README

**Người phụ trách:** 🟦 ___ · Làm **cuối cùng**, sau khi mọi mục trên chạy sạch.

### Danh sách kiểm tra trước khi nộp
- [ ] Runtime → **Run all** chạy sạch trong môi trường mới (không lỗi, `USE_SYNTHETIC=False`, `RUN_HEAVY` đúng ý).
- [ ] Mọi số liệu trong Thuyết minh/slides **khớp** các bảng trong notebook (số mẫu, tỷ lệ High, MAE, Recall…).
- [ ] Ngày truy cập hai nguồn dữ liệu **đã thống nhất** (Mục 1, 11).
- [ ] Đã chạy ✅ kiểm tra rò rỉ (Mục 5.7) – không assert nào lỗi.
- [ ] Đã xuất `DataInventory.xlsx`, các hình `fig_*.png`, `results/*.csv`.
- [ ] Đã đặt tên file: `Lab3_DS317.R11_Nhom5_Code`, `..._DataInventory`, `..._ThuyetMinh`, `..._Slides`, `..._PhanCong`.
- [ ] README (bên dưới) đã có link dữ liệu và hướng dẫn chạy.

In [ ]:
# ---- 13.1 Kiểm tra file đầu ra và ghi lại môi trường ----
import platform, importlib.metadata as im
expected = {'interim': ['es_sales_wk', 'es_stock_wk'], 'processed': ['mt_train', 'mt_val', 'mt_test'], 'results': ['res_reg.csv', 'res_clf.csv'], 'excel': ['Lab3_DS317.R11_Nhom5_DataInventory.xlsx']}
for folder, names in expected.items():
    for n in names:
        ok = any((DIRS[folder] / (n + e if '.' not in n else n)).exists() for e in ('.parquet', '.csv', '')) if '.' not in n else (DIRS[folder] / n).exists()
        print(('✅' if ok else '❌'), folder, n)
print('\nPython', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__)
for p_ in ('scikit-learn', 'lightgbm', 'torch'):
    try: print(p_, im.version(p_))
    except Exception: print(p_, '(chưa cài)')

### 13.2 Mẫu README *(🟦 sao chép ra file `README.md` và điền)*

```
# DS317 – Lab 3 – Nhóm 5 (DS317.R11)
Đề tài: Dự báo doanh số và cảnh báo rủi ro tồn kho theo SKU–tuần cho chuỗi bán lẻ điện tử bằng Transformer đa nhiệm

## Gói nộp
- Lab3_DS317.R11_Nhom5_Code        : notebook chung + NB1 (ES-bán hàng) + NB2 (ES-tồn kho) [+ NB3 Instacart]
- Lab3_DS317.R11_Nhom5_DataInventory.xlsx, ..._ThuyetMinh (PDF, DOCX), ..._Slides, ..._PhanCong

## Dữ liệu (không nộp kèm vì dung lượng lớn)
- Electronics Sales Dataset (CC0): <link Kaggle> – truy cập ngày <dd/mm/yyyy>
- Instacart Market Basket Analysis (CC0): <link Kaggle> – truy cập ngày <dd/mm/yyyy>
- Bản sao Drive: <link>
Đặt file .csv vào  DS317_Nhom5/raw/

## Cách chạy
1. Mở NB1 → chạy hết → file xuất vào interim/  (es_sales_wk, ...)
2. Mở NB2 → chạy hết → file xuất vào interim/  (es_stock_wk, ...)
3. Mở notebook chung → Run all (RUN_HEAVY=True để chạy Transformer, ~<x> phút trên Colab GPU)
Môi trường: Python 3.x, pandas, numpy, scikit-learn, lightgbm, torch, matplotlib, openpyxl
```